# Chakaso — Kaggle CPU thread scaling

**Question.** A Kaggle free-tier CPU notebook reaches **3,270 positions/s at 2 PyTorch threads**. Does raising
`torch.set_num_threads` to 4 buy anything on its 4 logical CPUs, or is 2 already at the machine's limit? The
answer decides whether E2636's six-arm session is ~11 hours or ~22 on that host, and E2636's record has no
measured rate for Kaggle at all.

**Honest status of the 3,270 figure.** That number is *not* in this repository — no committed file records a
Kaggle run — so it is carried here as the operator's reported starting point, and this notebook re-measures the
whole ladder rather than treating it as a baseline. The `2` in `THREAD_COUNTS` exists so the figure can be
reproduced or contradicted by this run, not to anchor a comparison on it.

**How it measures.** The setup, corpus discovery, fingerprint verification, packing and configuration cells are
**lifted verbatim** from
[`kaggle_cpu_benchmark.ipynb`](kaggle_cpu_benchmark.ipynb) — the same code, not a copy of it — so the two probes
are comparable by construction. The only thing that varies between arms is
`torch.set_num_threads(n)`, set immediately before each `train_transformer` call. Every arm trains a **fresh
model from the same seed** (`train_transformer` seeds and builds internally; the determinism of that path is
asserted below), on the **same packed blocks**, with the **same batch shape**, and reports the **median** of its
measured steps exactly as the first benchmark does.

**Why the ladder is walked twice, forwards then backwards.** The first arm of a process pays thread-pool
spin-up and first-touch of ~1.8 GiB of pages that later arms do not, so a single forward pass biases the
earliest thread count slow — the first benchmark measured that swing at 6,502 against 2,579 positions/s across
two runs of identical code. Pass 2 reverses the order, and the per-arm figure is the median across both.

**This is not E2636.** It launches no arm, writes no record, and touches no protocol file, seed list, band or
decision rule. It reads the E2636 protocol as an *input* to get the architecture and optimiser settings.
Its outputs land in `data/benchmarks/kaggle-cpu-threads/` locally (git-ignored) and `/kaggle/working/` on
Kaggle — never the first benchmark's files, and never `experiments/`. A guard cell asserts both.

## Running it on Kaggle

Identical setup to the first benchmark: upload the repository as a dataset, **Accelerator: None** (there is no
GPU code path in `chakaso.training`; the dependency cell asserts CUDA is absent), attach the pinned Gutenberg
snapshot only if you want the corpus fingerprint to be E2636's.

Wall clock, at Kaggle's reported 2-thread rate: `len(THREAD_COUNTS) × REPEATS × (WARMUP_STEPS + MEASURED_STEPS)`
updates, so 2 × 3 × 15 = 90 updates of 8,192 positions. At 3,270 positions/s that is ~3.7 minutes of gradient
work plus the 1-thread arm's slower share, plus corpus and tokenizer preparation; expect under ten minutes.
If you want it shorter, drop `REPEATS` to 1 — but the single forward pass is then order-biased, and the result
should say so.

`THREAD_COUNTS = (1, 2, 4)` is chosen against Kaggle's **4 logical CPUs**: 4 is the ceiling it can actually
use, 2 is the reported setting, 1 is the floor that shows how much of the rate is parallel at all. On a host
with a different topology change the tuple; the guard cell warns when a requested count exceeds
`os.cpu_count()`, because asking for 16 threads on a 4-CPU box measures the cgroup quota, not the code.

In [ ]:
from pathlib import Path

# --- the only cell an operator edits -------------------------------------------------
# Sizes are stated in the units the E2636 protocol uses, so the reduction is visible.
WARMUP_STEPS = 3  # gradient updates timed and discarded
MEASURED_STEPS = 12  # gradient updates that produce the reported rate
EVAL_BLOCKS = 128  # packed held-out examples scored once, for evaluation timing
TRAIN_TOKEN_CAP = 160_000  # identifiers packed for training (E2636 packs 14,033,424)
EVAL_TOKEN_CAP = 40_000  # identifiers packed for held-out timing
THREAD_COUNTS = (1, 2, 4)  # torch.set_num_threads values to compare, in ladder order
REPEATS = 2  # passes over the ladder; pass 2 walks it reversed, so ordering bias cancels
CORPUS_SNAPSHOT: Path | None = None  # or a path to gutenberg-multiregister-v1-snapshot.json
OUT_DIR: Path | None = None  # None -> /kaggle/working on Kaggle, data/benchmarks/kaggle-cpu here

In [ ]:
import importlib.metadata
import json
import os
import platform
import shutil
import socket
import statistics
import sys
import time

import torch

KAGGLE = Path("/kaggle/working").is_dir()
SNAPSHOT_NAME = "gutenberg-multiregister-v1-snapshot.json"


def _find_repo_root() -> Path:
    """Find the checkout by marker, not by a mount path.

    On Kaggle an uploaded dataset lands at /kaggle/input/<dataset>/ and the notebook's cwd is /kaggle/working;
    locally the notebook sits inside the repository. The marker is the training module's own file, because a
    directory containing it is the project and a directory with a similar name is not.
    """
    marker = Path("src/chakaso/training/nn_training.py")
    mounts = sorted(p for p in Path("/kaggle/input").glob("*") if p.is_dir()) if KAGGLE else []
    for seed in [Path.cwd(), *mounts, *(d for m in mounts for d in sorted(m.glob("*"))[:24])]:
        for candidate in (seed, seed / "chakaso", seed / "chakaso-main"):
            if (candidate / marker).is_file():
                return candidate.resolve()
    message = (
        "no Chakaso checkout found: looked for src/chakaso/training/nn_training.py under "
        f"{Path.cwd()} and under /kaggle/input. Upload the repository as a dataset."
    )
    raise FileNotFoundError(message)


REPO = _find_repo_root()
# `import chakaso` must resolve to THIS checkout, not to whatever wheel the image happens to carry: a
# benchmark measured against an installed copy of different code would be a number about nothing.
sys.path.insert(0, str(REPO / "src"))

print(f"repository root : {REPO}")
print(f"kaggle          : {KAGGLE}")
print(f"host            : {socket.gethostname()} on {platform.platform()}")
print(f"python          : {sys.version.split()[0]} at {sys.executable}")

In [ ]:
import chakaso
from chakaso.training.nn_checkpoint import describe_environment

environment = describe_environment()
for name, value in sorted(environment.items()):
    print(f"{name:<15}: {value}")

for distribution in ("chakaso", "numpy", "torch"):
    try:
        version = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        version = "ABSENT -- pip install it before continuing"
    print(f"dist {distribution:<12}: {version}")

print(f"package file    : {Path(chakaso.__file__).resolve()}")
assert environment["device"] == "cpu"
assert not torch.cuda.is_available(), (
    "a CUDA device is visible; this probe must run on CPU to compare"
)
assert Path(chakaso.__file__).resolve().is_relative_to(REPO), (
    "the imported chakaso is not the checkout under test -- the sys.path insert above did not take"
)

In [ ]:
from chakaso.core.process_memory import available_memory, peak_memory

machine = available_memory()
before = peak_memory()
disk_target = Path("/kaggle/working") if KAGGLE else REPO
disk = shutil.disk_usage(disk_target)
print(f"logical cpus (os)  : {os.cpu_count()}")
print(f"torch threads      : {torch.get_num_threads()}")
print(f"torch interop      : {torch.get_num_interop_threads()}")
free = machine.available_bytes
print(
    f"physical memory    : {(machine.total_bytes or 0) / 2**30:.2f} GiB total, "
    f"{(free or 0) / 2**30:.2f} GiB free  [{machine.source}]"
)
print(f"disk at output     : {disk.free / 2**30:.1f} GiB free of {disk.total / 2**30:.1f} GiB")
print(
    f"process peak (now) : {(before.peak_resident_bytes or 0) / 2**20:.1f} MiB  [{before.source}]"
)
print(f"cpu model          : {platform.processor() or platform.machine()}")

In [ ]:
OUT_DIR = OUT_DIR or (
    Path("/kaggle/working") if KAGGLE else REPO / "data/benchmarks/kaggle-cpu-threads"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = OUT_DIR.resolve()

# Three things this probe must not do: write beside a committed record, overwrite the first benchmark's
# results, or quietly replace them if it happens to share a working directory with it.
assert (REPO / "experiments").resolve() not in OUT_DIR.parents, (
    f"refusing to write benchmark output inside the committed records at {OUT_DIR}"
)
RESULT_NAME = "kaggle-cpu-thread-scaling"
for forbidden in ("kaggle-cpu-benchmark.json", "kaggle-cpu-benchmark.md"):
    assert (OUT_DIR / forbidden).resolve() != (OUT_DIR / f"{RESULT_NAME}.json").resolve()
    assert (OUT_DIR / forbidden).resolve() != (OUT_DIR / f"{RESULT_NAME}.md").resolve()
FIRST_BENCHMARK = REPO / "experiments/benchmarks/kaggle_cpu_benchmark.ipynb"
assert FIRST_BENCHMARK.is_file(), "the benchmark this one inherits its methodology from is missing"

requested_over = [n for n in THREAD_COUNTS if os.cpu_count() and n > os.cpu_count()]
if requested_over:
    print(
        f"WARNING: THREAD_COUNTS {tuple(requested_over)} exceed os.cpu_count()={os.cpu_count()}; "
        "the run will measure the scheduler's ceiling, not the code's scaling"
    )
print(f"output directory  : {OUT_DIR}")
print(f"results named     : {RESULT_NAME}.json / {RESULT_NAME}.md")
print(f"ladder            : {THREAD_COUNTS} x {REPEATS} pass(es), pass 2 reversed")

## Fingerprints, read from the committed artifacts

Both vocabularies E2636 declares are **committed in the repository**, so they ship with the upload and their
identities are verified here against the values in the protocol file — not assumed from a filename.
`SubwordVocabulary.load` re-checks the file's own checksum before returning, so a truncated or edited merge
table fails here rather than becoming a benchmark result.

In [ ]:
from chakaso.training.subword import SubwordVocabulary

TOKENIZER_FILE = REPO / "experiments/E2620/vocabulary-4a47f0281c81a9e5.json"
SELECTION_TOKENIZER_FILE = REPO / "experiments/E2612/vocabulary-434debe82f949347.json"

vocabulary = SubwordVocabulary.load(TOKENIZER_FILE)
selection_vocabulary = SubwordVocabulary.load(SELECTION_TOKENIZER_FILE)

print(
    f"training vocabulary     : {vocabulary.fingerprint}  size={vocabulary.size} "
    f"min_frequency={vocabulary.min_frequency}"
)
print(f"  pre_tokenizer         : {vocabulary.pre_tokenizer}")
print(f"  file                  : {TOKENIZER_FILE.relative_to(REPO)}")
print(
    f"evaluation-selection    : {selection_vocabulary.fingerprint}  "
    f"size={selection_vocabulary.size}"
)
print(f"  pre_tokenizer         : {selection_vocabulary.pre_tokenizer}")

assert vocabulary.fingerprint == "4a47f0281c81a9e5", (
    f"E2636's pinned training vocabulary is 4a47f0281c81a9e5, loaded {vocabulary.fingerprint}"
)
assert selection_vocabulary.fingerprint == "434debe82f949347", (
    f"E2636's pinned selection vocabulary is 434debe82f949347, loaded "
    f"{selection_vocabulary.fingerprint}"
)

## Corpus, and what its fingerprint proves

`read_snapshot` re-hashes every document in the snapshot against the manifest it carries, so a fingerprint that
equals E2636's is a statement about 149,000,499 characters of bytes, not about a filename. If no snapshot is
attached the cell falls back to this repository's tracked markdown, through the same `build_manifest` the pinned
`chakaso-repo-text-v1` corpus was made with, and says so in the result file.

In [ ]:
from chakaso.training.manifest import build_manifest, read_snapshot
from chakaso.training.subword_experiment import REPO_TEXT_CORPUS

E2636_CORPUS_FINGERPRINT = "312571f9fa8351a8"


def _find_snapshot() -> Path | None:
    if CORPUS_SNAPSHOT is not None:
        return Path(CORPUS_SNAPSHOT)
    hits = sorted(Path("/kaggle/input").rglob(SNAPSHOT_NAME)) if KAGGLE else []
    hits += sorted(REPO.rglob(SNAPSHOT_NAME))
    return hits[0] if hits else None


snapshot_path = _find_snapshot()
if snapshot_path is not None:
    snapshot = read_snapshot(snapshot_path)
    manifest = snapshot.manifest
    corpus_origin = "the pinned E2636 snapshot, re-hashed document by document"
    fingerprint_matches = manifest.fingerprint == E2636_CORPUS_FINGERPRINT
    if not fingerprint_matches:
        print(
            f"WARNING: snapshot at {snapshot_path} hashes to {manifest.fingerprint}, "
            f"not E2636's {E2636_CORPUS_FINGERPRINT} -- this is a different corpus"
        )
else:
    manifest = build_manifest(
        REPO,
        name=REPO_TEXT_CORPUS.name,
        version=REPO_TEXT_CORPUS.version,
        licence=REPO_TEXT_CORPUS.licence,
        source=REPO_TEXT_CORPUS.source,
        include=REPO_TEXT_CORPUS.include,
        extensions=REPO_TEXT_CORPUS.extensions,
    )
    snapshot, corpus_origin = None, "this repository's tracked markdown (NOT E2636's corpus)"
    fingerprint_matches = False

units = manifest.text_units()
print(f"corpus source           : {corpus_origin}")
print(f"corpus name/version     : {manifest.name} {manifest.version}")
print(f"fingerprint             : {manifest.fingerprint}")
print(f"== E2636's declared     : {fingerprint_matches}")
print(f"documents               : {len(manifest.documents)}  characters: {manifest.char_count:,}")
print(f"split-ready units       : {len(units)}")

In [ ]:
from chakaso.training.data import split_text_units
from chakaso.training.packing import pack_units, select_units_by_token_cap


def _capped_sample(units, reference, cap, *, minimum_works=3, stride=1):
    """Whole units from across the corpus, under ``cap``, spanning at least ``minimum_works`` works.

    E2636 splits by whole work at split seed 7, and `split_text_units` refuses a whole-work split over fewer
    than three works. Capping in manifest order would take the corpus's first 37 chunks -- two books, because
    a book's chunks are adjacent -- so the benchmark strides the corpus before capping. The units are fewer,
    they come from across the whole corpus, and the rule that assigns sides is the record's own.
    """
    while True:
        chosen = select_units_by_token_cap(units[::stride], reference, max_tokens=cap)
        works = {unit.group for unit in chosen}
        if len(works) >= minimum_works or stride >= len(units):
            return chosen, stride, works
        stride *= 4


sampled, stride, works = _capped_sample(units, vocabulary, TRAIN_TOKEN_CAP + EVAL_TOKEN_CAP)
# E2636's split rule, not a convenience one: whole-work separation at split seed 7.
split = split_text_units(
    sampled, validation_fraction=0.15, test_fraction=0.15, seed=7, whole_works=True
)
train_pack = pack_units(split.train, vocabulary, context_length=256)
held_out = pack_units(split.test, vocabulary, context_length=256)
print(f"stride over the corpus  : every {stride}th unit")
print(f"units sampled           : {len(sampled)} from {len(units)}, across {len(works)} works")
print(
    f"train / validation / test units: {len(split.train)} / {len(split.validation)} / {len(split.test)}"
)
print(f"dropped duplicates      : {len(split.dropped_duplicates)}")
print(
    f"train blocks            : {len(train_pack.examples)} of 256 positions "
    f"({train_pack.tokens_available:,} identifiers)"
)
print(f"held-out blocks         : {len(held_out.examples)}")
print(f"split fingerprint       : {split.fingerprint}")
assert train_pack.examples and held_out.examples, "nothing packed: the corpus selects no text"
leaked = {u.identifier for u in split.train} & {u.identifier for u in split.test}
assert not leaked, f"leakage: the same document is on both sides: {sorted(leaked)[:3]}"

## Declared versus reduced

The configuration is **read from E2636's own protocol file** through `chakaso.training.protocol.load_plan`, the
loader the experiment uses, and passed through `architecture_for` and `_training_config` — the same two functions
`run_experiment` calls. Nothing below is retyped; the table prints what the record declares against what this
benchmark runs, so a reduction is visible instead of smuggled.

In [ ]:
from dataclasses import replace

from chakaso.training.protocol import load_plan
from chakaso.training.subword_experiment import _training_config, architecture_for

PROTOCOL = REPO / "configs/experiments/E2636-adopted-budget-7200.toml"
plan = load_plan(
    PROTOCOL,
    root=REPO,
    work_dir=OUT_DIR,
    fixed={
        "split_seed": 7,
        "expected_corpus_fingerprint": E2636_CORPUS_FINGERPRINT,
        "corpus_snapshot": snapshot_path,
    },
)
architecture = architecture_for(plan, vocabulary)
declared = _training_config(plan, architecture)

# The benchmark's own ceiling: warm-up then measurement, one evaluation per step so the loop's own
# per-interval rate is reported at step granularity. positions_per_second is measured over gradient work
# ONLY (nn_training's definition), so evaluating every step does not inflate it.
steps = WARMUP_STEPS + MEASURED_STEPS
reduced = replace(
    declared,
    max_steps=steps,
    eval_every=1,
    warmup_steps=min(declared.warmup_steps, max(1, steps // 2)),
    checkpoint_every=steps,
)

REDUCED_FIELDS = ("max_steps", "eval_every", "warmup_steps", "checkpoint_every")
print(f"{'field':<24}{'E2636 declares':>18}{'this benchmark':>18}")
for name in REDUCED_FIELDS:
    print(f"{name:<24}{getattr(declared, name)!s:>18}{getattr(reduced, name)!s:>18}")
UNCHANGED = (
    "architecture",
    "batch_size",
    "grad_accum_steps",
    "learning_rate",
    "weight_decay",
    "schedule",
    "decay_to",
    "gradient_clip_norm",
    "patience",
    "min_delta",
    "document_masking",
    "seed",
)
print(f"\n{'unchanged from the record ->':<24}")
for name in UNCHANGED:
    print(f"{name:<24}{getattr(reduced, name)!s:>18}")

assert reduced.architecture == declared.architecture == architecture
assert architecture.parameter_count == 3_413_760, (
    f"E2636's reference configuration is 3,413,760 parameters, got {architecture.parameter_count:,}"
)
assert reduced.batch_size == 32 and reduced.max_steps == steps
print(f"\nparameter count       : {architecture.parameter_count:,}")
print(f"positions per step    : {reduced.batch_size * architecture.context_length:,}")
print(
    f"projected gradient work: {steps * reduced.batch_size * architecture.context_length:,} positions"
)

## Each arm must differ only in threading

`train_transformer` builds its own model when none is passed: `torch.manual_seed(config.seed)` then
`build_model(config.architecture)` (`nn_training.py:823`). So a fresh call under the same configuration starts
from identical parameters — but that is asserted, not assumed, because an arm that quietly started from a
different model would be measuring two things at once.

In [ ]:
from chakaso.training.nn_training import build_model

# Same seed, same architecture, twice: the parameters must be bit-identical before any threading is involved.
torch.manual_seed(reduced.seed)
first = build_model(reduced.architecture)
torch.manual_seed(reduced.seed)
second = build_model(reduced.architecture)
first_state, second_state = first.state_dict(), second.state_dict()
assert first_state.keys() == second_state.keys(), (
    "two builds of one config exposed different tensors"
)
identical = all(
    torch.equal(stored, replayed)
    for stored, replayed in zip(first_state.values(), second_state.values(), strict=True)
)
print(f"fresh-model init identical across two seeded builds: {identical}")
assert identical, "the seed does not pin initialization, so arms would not be comparable"
print(f"seed under test       : {reduced.seed}")
print(
    f"batch / context       : {reduced.batch_size} x {reduced.architecture.context_length} "
    f"= {reduced.batch_size * reduced.architecture.context_length:,} positions per update"
)
print(
    f"train blocks available: {len(train_pack.examples)}  held-out blocks: {len(held_out.examples)}"
)

## One arm, and the ladder

Each arm sets the thread count, trains a fresh model on the same packed blocks for `WARMUP_STEPS +
MEASURED_STEPS` updates with `eval_every = 1`, then scores the same held-out blocks. The reported rate is the
**median of the measured steps** — the same statistic the first benchmark projects from, and for the same
reason: the loop's `positions_per_second` is measured over gradient work only, so the per-step series is the
clean signal and the first steps of a process are not.

In [ ]:
from chakaso.core.process_memory import current_memory
from chakaso.training.nn_evaluation import evaluate_model
from chakaso.training.nn_training import train_transformer

POSITIONS_PER_STEP = reduced.batch_size * reduced.architecture.context_length
EVAL_EXAMPLES = held_out.examples[:EVAL_BLOCKS]


def run_arm(threads: int):
    """One thread setting: set it, train fresh, then time the held-out pass. Nothing else varies."""
    torch.set_num_threads(threads)
    if torch.get_num_threads() != threads:
        print(f"  note: asked for {threads} threads, torch reports {torch.get_num_threads()}")
    resident_before = (current_memory().resident_bytes or 0) / 2**20

    started = time.perf_counter()
    outcome = train_transformer(
        train_examples=train_pack.examples,
        validation_examples=held_out.examples,
        vocabulary=vocabulary,
        config=reduced,
        corpus_fingerprint=manifest.fingerprint,
        experiment="chakaso-threadbench",
        split_fingerprint=split.fingerprint,
    )
    run_seconds = time.perf_counter() - started

    per_step = [r for r in outcome.records if r.step > 0]
    step_rates = [r.positions_per_second for r in per_step]
    warm = step_rates[:WARMUP_STEPS]
    measured = step_rates[WARMUP_STEPS:]
    # Gradient time is recoverable from the per-step rates, so what is left over is the in-run evaluation and
    # bookkeeping cost -- the overhead a 7,200-step arm pays 19 times instead of 7.
    gradient_seconds = sum(POSITIONS_PER_STEP / rate for rate in step_rates)

    forward = time.perf_counter()
    with torch.no_grad():
        metrics = evaluate_model(
            outcome.artifact().to_model(), EVAL_EXAMPLES, vocabulary, batch_size=reduced.batch_size
        )
    eval_seconds = time.perf_counter() - forward

    return {
        "threads": torch.get_num_threads(),
        "asked_for": threads,
        "median_positions_per_second": statistics.median(measured),
        "mean_positions_per_second": sum(measured) / len(measured),
        "min_positions_per_second": min(measured),
        "max_positions_per_second": max(measured),
        "warmup_rates": warm,
        "measured_rates": measured,
        "run_wall_clock_seconds": run_seconds,
        "in_run_overhead_seconds": run_seconds - gradient_seconds,
        "in_run_overhead_per_evaluation": (run_seconds - gradient_seconds) / len(per_step),
        "evaluation_seconds": eval_seconds,
        "evaluation_positions_per_second": metrics.positions_scored / eval_seconds,
        "held_out_loss_nats": metrics.loss_nats,
        "held_out_bits_per_character": metrics.bits_per_character,
        "token_accuracy": metrics.accuracy,
        "final_validation_loss_nats": outcome.final_validation_loss,
        "artifact_checksum": outcome.artifact().checksum()[:16],
        "resident_before_mib": resident_before,
        "resident_after_mib": (current_memory().resident_bytes or 0) / 2**20,
        "process_peak_mib": (peak_memory().peak_resident_bytes or 0) / 2**20,
    }


arms: list[dict] = []
for repetition in range(REPEATS):
    order = THREAD_COUNTS if repetition % 2 == 0 else tuple(reversed(THREAD_COUNTS))
    print(f"--- pass {repetition + 1}: thread ladder {order}")
    for threads in order:
        arm = run_arm(threads)
        arm["pass"] = repetition + 1
        arms.append(arm)
        print(
            f"  {arm['threads']:>2} threads: median {arm['median_positions_per_second']:>8,.0f} pos/s "
            f"(spread {arm['min_positions_per_second']:,.0f}-{arm['max_positions_per_second']:,.0f}) "
            f"run {arm['run_wall_clock_seconds']:>6.1f} s  eval {arm['evaluation_seconds']:>5.2f} s  "
            f"resident {arm['resident_after_mib']:>6,.0f} MiB  peak {arm['process_peak_mib']:>6,.0f} MiB"
        )

print()
print(
    "Fingerprints are the same for every arm by construction: one corpus, one tokenizer, one split."
)
print(
    f"  corpus {manifest.fingerprint}  tokenizer {vocabulary.fingerprint}  split {split.fingerprint}"
)

In [ ]:
# Collapse the passes: every measured step of a thread count is one sample pool, so the reported figure is
# not whichever pass happened to land on a quieter machine.
by_threads: dict[int, dict] = {}
for threads in THREAD_COUNTS:
    passes = [arm for arm in arms if arm["asked_for"] == threads]
    if not passes:
        continue
    pooled = [rate for arm in passes for rate in arm["measured_rates"]]
    median_rate = statistics.median(pooled)
    eval_rate = statistics.median([arm["evaluation_positions_per_second"] for arm in passes])
    one_evaluation = 250_000 / eval_rate
    gradient = 3 * (2400 + 7200) * POSITIONS_PER_STEP / median_rate
    by_threads[threads] = {
        "threads": threads,
        "passes": len(passes),
        "measured_steps": len(pooled),
        "median_positions_per_second": round(median_rate, 1),
        "mean_positions_per_second": round(sum(pooled) / len(pooled), 1),
        "min_positions_per_second": round(min(pooled), 1),
        "max_positions_per_second": round(max(pooled), 1),
        "stdev_positions_per_second": round(statistics.stdev(pooled), 1)
        if len(pooled) > 1
        else None,
        "per_pass_medians": [round(arm["median_positions_per_second"], 1) for arm in passes],
        "evaluation_positions_per_second": round(eval_rate, 1),
        "in_run_overhead_seconds": round(
            statistics.median([arm["in_run_overhead_seconds"] for arm in passes]), 2
        ),
        "in_run_overhead_per_evaluation_s": round(
            statistics.median([arm["in_run_overhead_per_evaluation"] for arm in passes]), 2
        ),
        "one_evaluation_at_250000_s": round(one_evaluation, 1),
        "resident_after_mib": round(
            statistics.median([a["resident_after_mib"] for a in passes]), 0
        ),
        "process_peak_mib": round(max(a["process_peak_mib"] for a in passes), 0),
        "held_out_bits_per_character": round(
            statistics.median([a["held_out_bits_per_character"] for a in passes]), 5
        ),
        "artifact_checksums": sorted({a["artifact_checksum"] for a in passes}),
        "projection_for_e2636_hours": {
            "gradient_only": round(gradient / 3600, 2),
            "including_record_evaluation_load": round(
                (gradient + 3 * 26 * one_evaluation) / 3600, 2
            ),
            "short_arm": round(2400 * POSITIONS_PER_STEP / median_rate / 3600, 2),
            "long_arm": round(7200 * POSITIONS_PER_STEP / median_rate / 3600, 2),
        },
    }

fewest = min(by_threads.values(), key=lambda row: row["threads"])["median_positions_per_second"]
for row in by_threads.values():
    row["speedup_vs_fewest_threads"] = round(row["median_positions_per_second"] / fewest, 2)
    row["scaling_efficiency"] = round(row["speedup_vs_fewest_threads"] / row["threads"], 2)

print(
    f"{'thr':>4}{'median pos/s':>14}{'spread':>15}{'stdev':>8}{'speedup':>9}{'per-core':>9}"
    f"{'eval pos/s':>12}{'6 arms h':>10}"
)
for threads in THREAD_COUNTS:
    if threads not in by_threads:
        continue
    row = by_threads[threads]
    spread = f"{row['min_positions_per_second']:,.0f}-{row['max_positions_per_second']:,.0f}"
    print(
        f"{threads:>4}{row['median_positions_per_second']:>14,.0f}{spread:>15}"
        f"{(row['stdev_positions_per_second'] or 0):>8,.0f}{row['speedup_vs_fewest_threads']:>9}"
        f"{row['scaling_efficiency']:>9}{row['evaluation_positions_per_second']:>12,.0f}"
        f"{row['projection_for_e2636_hours']['including_record_evaluation_load']:>10}"
    )

checksums = {row["artifact_checksums"][0] for row in by_threads.values()}
print(
    f"\nthread count -> trained model outcome differs: {len(checksums) > 1} "
    f"({len(checksums)} distinct artifact checksums across {len(by_threads)} settings)"
)

In [ ]:
LIMITATIONS = [
    "One process for the whole ladder: arms share page cache, allocator state and a rising process-wide "
    "memory high-water mark. Walking the ladder reversed on the second pass counters ordering bias; it "
    "does not eliminate it, and REPEATS=1 leaves it in.",
    "Peak memory is not attributable to one arm. The platform reports a since-process-start high-water "
    "mark, so each row carries its own current residency plus the run's overall peak.",
    "Small corpus by design. The text is a stride-sampled subset sized to keep the probe short, its split "
    "fingerprint is that sample's own, and neither is E2636's whole-corpus split. Throughput is set by the "
    "fixed 32 x 256 batch shape, which is the record's shape; the loss figures are not comparable to any "
    "measured record.",
    "Steady state, not a long run. A few dozen updates measure matmul and optimiser throughput; they do not "
    "measure the cache, memory-bandwidth or allocator behaviour of 7,200 steps over 149,000,499 characters.",
    "One batch shape only. It is the shape E2636 runs, so it is the right one, but nothing here says how a "
    "different shape scales with threads.",
    "Host topology differs. Logical CPUs on this machine are hyperthread siblings of physical cores under no "
    "cgroup quota; a Kaggle CPU notebook's 4 logical CPUs are a quota on shared hardware. A thread ladder run "
    "here predicts the shape of the curve, not its value there -- which is why this notebook is meant to be "
    "run on Kaggle, not only locally.",
    "Every E2636 figure is arithmetic on a probe, not a measurement of E2636. No arm was launched.",
]

result = {
    "schema": "chakaso-kaggle-cpu-thread-scaling/1",
    "question": (
        "does raising torch.set_num_threads beyond 2 improve Chakaso training throughput on a 4 logical CPU "
        "host, and by how much"
    ),
    "starting_point": {
        "reported_positions_per_second_at_2_threads": 3270,
        "provenance": (
            "the operator's Kaggle observation, stated to this session from outside the repository. It is NOT "
            "recorded in any committed file and is not treated as a baseline: thread count 2 is re-measured "
            "here so the comparison rests on this run."
        ),
    },
    "is_not_e2636": "a thread ladder over a few dozen updates; no E2636 arm was launched by it",
    "methodology_inherited_from": "experiments/benchmarks/kaggle_cpu_benchmark.ipynb",
    "environment": environment,
    "machine": {
        "hostname": socket.gethostname(),
        "platform": platform.platform(),
        "cpu_model": platform.processor() or platform.machine(),
        "logical_cpus": os.cpu_count(),
        "interop_threads": torch.get_num_interop_threads(),
        "physical_total_bytes": machine.total_bytes,
        "physical_available_bytes": machine.available_bytes,
        "memory_source": machine.source,
        "disk_free_bytes": disk.free,
    },
    "settings": {
        "thread_counts": list(THREAD_COUNTS),
        "repeats": REPEATS,
        "warmup_steps_excluded": WARMUP_STEPS,
        "measured_steps_per_pass": MEASURED_STEPS,
        "positions_per_step": POSITIONS_PER_STEP,
        "batch_size": reduced.batch_size,
        "context_length": reduced.architecture.context_length,
        "architecture": reduced.architecture.as_dict(),
        "parameter_count": reduced.architecture.parameter_count,
        "seed": reduced.seed,
        "train_blocks": len(train_pack.examples),
        "eval_blocks": len(EVAL_EXAMPLES),
        "reduced_from_e2636": {name: getattr(reduced, name) for name in REDUCED_FIELDS},
    },
    "fingerprints": {
        "training_tokenizer": vocabulary.fingerprint,
        "selection_tokenizer": selection_vocabulary.fingerprint,
        "corpus": manifest.fingerprint,
        "corpus_is_e2636_pinned": fingerprint_matches,
        "corpus_origin": corpus_origin,
        "split": split.fingerprint,
    },
    "per_thread_count": {str(key): value for key, value in sorted(by_threads.items())},
    "arms": [
        {
            key: (round(value, 3) if isinstance(value, float) else value)
            for key, value in arm.items()
            if key != "measured_rates"
        }
        | {"measured_rates": [round(v, 1) for v in arm["measured_rates"]]}
        for arm in arms
    ],
    "limitations": LIMITATIONS,
}

result_path = OUT_DIR / f"{RESULT_NAME}.json"
result_path.write_text(json.dumps(result, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(f"wrote {result_path}  ({result_path.stat().st_size:,} bytes)")

In [ ]:
summary_path = OUT_DIR / f"{RESULT_NAME}.md"
best = max(by_threads.values(), key=lambda row: row["median_positions_per_second"])
at_two = by_threads.get(2)
rows = []
for threads in THREAD_COUNTS:
    if threads not in by_threads:
        continue
    row = by_threads[threads]
    hours = row["projection_for_e2636_hours"]
    rows.append(
        f"| {threads} | {row['median_positions_per_second']:,.0f} | "
        f"{row['min_positions_per_second']:,.0f}-{row['max_positions_per_second']:,.0f} | "
        f"{row['stdev_positions_per_second'] or 0:,.0f} | {row['speedup_vs_fewest_threads']}x | "
        f"{row['scaling_efficiency']} | {row['evaluation_positions_per_second']:,.0f} | "
        f"{row['process_peak_mib']:,.0f} | {hours['including_record_evaluation_load']} h |"
    )

lines = [
    "# Chakaso CPU thread scaling — result",
    "",
    f"Host `{result['machine']['hostname']}` on `{result['machine']['platform']}`, "
    f"{result['machine']['logical_cpus']} logical CPUs, torch {result['environment']['torch']}, "
    f"{REPEATS} pass(es) over {THREAD_COUNTS}.",
    "",
    f"**Answer:** {best['threads']} thread(s) is the best setting measured, at "
    f"{best['median_positions_per_second']:,.0f} positions/s — "
    f"{best['speedup_vs_fewest_threads']}x the {min(by_threads)}-thread arm and "
    f"{best['scaling_efficiency']} of ideal per-core scaling.",
    "",
    "| threads | median pos/s | spread | stdev | speedup | per-core | eval pos/s | peak MiB | 6 arms |",
    "| --- | --- | --- | --- | --- | --- | --- | --- | --- |",
    *rows,
    "",
    f"`6 arms` projects E2636's six declared arms (3 x 2,400 + 3 x 7,200 updates of "
    f"{POSITIONS_PER_STEP:,} positions) at each row's own rate, including the record's evaluation load "
    f"at that row's measured forward throughput. It is arithmetic on a probe: **no E2636 arm was run here**.",
    "",
    "**Starting figure.** "
    + (
        f"Thread count 2 measured {at_two['median_positions_per_second']:,.0f} pos/s here against the 3,270 "
        "pos/s reported from Kaggle."
        if at_two
        else "Thread count 2 was not in THREAD_COUNTS, so the reported Kaggle figure went untested here."
    )
    + " That 3,270 came from outside this repository — no committed file records a Kaggle run — so it is a "
    "claim to test, not a baseline to compare against.",
    "",
    f"**Are the arms the same experiment?** Distinct artifact checksums across thread settings: "
    f"{len({r['artifact_checksums'][0] for r in by_threads.values()})}. A reduction order that changes with "
    "the thread count changes the trained weights slightly, which is why nn_checkpoint records the thread "
    "count in the environment block of every artifact.",
    "",
    "## Limitations, stated rather than footnoted",
    "",
    *[f"- {text}" for text in LIMITATIONS],
    "",
    f"_Schema `{result['schema']}`; machine-readable copy beside this file. Fingerprints — corpus "
    f"`{result['fingerprints']['corpus']}`, tokenizer `{result['fingerprints']['training_tokenizer']}`, "
    f"split `{result['fingerprints']['split']}`._",
]
summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"wrote {summary_path}")
print()
print("\n".join(lines))

## Reading the result

- **`per-core` (scaling efficiency)** is the number that answers the question actually asked. Ideal scaling
  is 1.0; a laptop with hyperthread siblings commonly lands near 0.5–0.7 per core because the siblings share
  one physical core's execution units. A 4-thread row at, say, 2.1x over 1 thread is a real win that is *not*
  a 4x win, and reading only the speedup would oversell it.
- **`spread` and `stdev`** are computed over every measured step of every pass — `REPEATS × MEASURED_STEPS`
  samples — not over passes. With twelve steps per pass the distribution is thin; the two passes exist to
  show that, not to hide it. If the per-pass medians disagree by more than a few percent, the ladder is
  noise-sensitive on that host and the ordering effect is not yet cancelled: raise `REPEATS` before believing
  a projection.
- **`eval pos/s`** is forward-only throughput on the held-out pack, and it scales differently from gradient
  work — no backward pass, no optimizer step, so less to parallelise. It prices the projection's evaluation
  term; it is not E2636's held-out score, which is measured on a far larger pack.
- **`peak MiB`** is the process-wide high-water mark, not a per-arm figure: the platform cannot attribute it.
  Compare it against the host's total to judge whether the free tier holds the run at all.
- **Decide on the Kaggle run, not this one.** Thread scaling depends on how many physical cores sit under the
  logical CPUs and on cgroup quota, neither of which a laptop reproduces. This ladder on Kaggle is the
  measurement; the same ladder here is the shape check.

**Next, if 4 threads is a real gain and 2 was Kaggle's default.** The actionable follow-up is not a longer
E2636 session on a 2-thread setting — it is to set `torch.set_num_threads` deliberately at the start of the
run and record the value in the experiment's environment block, which `describe_environment()` already
captures. Whether to pin it, and to what, is a decision for the record that runs E2636; this notebook only
supplies the curve.